# Optional semantic models

FreshData's default install is **model-free** — everything you saw in the
context-cleaning and plan-approval notebooks is deterministic, offline code.
This notebook shows the *optional* Phase-3 model path: a small local ONNX
encoder that adds repair **proposals** (never decisions) for values the
deterministic experts abstain on, plus a rescue rung for the column resolver.

Requirements: `pip install "freshdata-cleaner[semantic]"` and an explicit
`fd.models.pull(...)`. Models are **never** downloaded automatically. For this
notebook we enable the built-in deterministic *stub encoder* so everything
runs with no model files and no network.


In [ ]:
import os

# Testing seam: a deterministic hash-based encoder, so the whole notebook
# runs offline with zero model files. With real models installed you would
# skip this line entirely.
os.environ["FRESHDATA_STUB_ENCODER"] = "1"

import pandas as pd

import freshdata as fd


## The model registry

`fd.models` is the explicit, auditable way models get onto your machine.
`status()` is offline and side-effect free; `pull()` is the **only** network
path in the whole package and verifies pinned checksums.


In [ ]:
for cfg in fd.models.list_available():
    print(f"{cfg.model_id:<20} {cfg.quantization:<5} {cfg.license:<12} "
          f"~{cfg.size_bytes / 1e6:.0f} MB")


In [ ]:
fd.models.status()


Official artifacts are not hosted yet — `fd.models.pull("fd-col-encoder-v1")`
raises `ModelNotPublishedError` with instructions (mirror via
`FRESHDATA_MODEL_URL_BASE`, or drop files into `FRESHDATA_MODEL_DIR` for
air-gapped installs). The stub we enabled above stands in for the encoder.


## Embedding-assisted cleaning

The ecommerce fixture again — but note `status` contains `'activvee'`, two edits
away from `active`. The deterministic reference expert only trusts one edit, so
on a model-free install this value is *flagged, never repaired*. With the
embedding backend enabled, the encoder gets a vote — as **evidence through the
same gate**, never as authority.


In [ ]:
df = pd.DataFrame(
    {
        "cust_id": ["C001", "C002", "C003", "C004"],
        "email_addr": ["asha@gmail.com", "ravi@@example.com",
                       "neha@shop.in", "kiran@test.in"],
        "mob_no": ["98765 43210", "+91 91234 56780",
                   "09123456789", "90000 11111"],
        "monthly_revenue": ["1000", "2000", "3000", "4000"],
        "status": ["activvee", "inactive", "pendng", "active"],
    }
)

CONTEXT = """
This is an ecommerce customer dataset.
CustomerID is unique.
Emails must be valid.
Phone numbers are Indian.
Allowed status values are active, inactive, pending.
Never modify revenue values.
"""

clean_df, report = fd.clean(
    df,
    context=CONTEXT,
    semantic_mode="auto",
    semantic_backends=("deterministic", "memory", "embedding"),
    return_report=True,
)
clean_df


Protected means protected — `monthly_revenue` is byte-identical no matter
which backends run (a physical executor check, not a convention):


In [ ]:
assert clean_df["monthly_revenue"].equals(df["monthly_revenue"])
clean_df["monthly_revenue"]


Every model-assisted action carries full provenance: which backend, which
model, the raw score, the calibrated confidence, and the candidate margins:


In [ ]:
for action in report.actions:
    if action.step == "semantic":
        meta = action.metadata
        print(f"[{action.status:<9}] {action.column:<12} "
              f"{meta.get('raw_value')!r} -> {meta.get('proposed_value')!r} "
              f"conf={action.confidence} backend={meta.get('backend')} "
              f"calib={meta.get('calibration_version', '-')}")


## Graceful degradation

Requesting the embedding backend without the extra or the model never crashes
— it self-disables with a report note, and cleaning proceeds exactly as the
model-free install would:


In [ ]:
import freshdata.models.runtime as runtime

os.environ.pop("FRESHDATA_STUB_ENCODER", None)
runtime.reset_encoders()

_, degraded_report = fd.clean(
    df,
    context=CONTEXT,
    semantic_mode="auto",
    semantic_backends=("deterministic", "memory", "embedding"),
    return_report=True,
)
degraded_report.fallback_events


## What the model path will never do

- **No automatic downloads** — `fd.clean` cannot reach the network.
- **No LLM, no cloud inference, no per-cell model calls** — the backend sees
  distinct values of eligible columns only.
- **No ambiguous auto-applies** — a value near two allowed candidates produces
  *no proposal at all*; pure similarity clustering is suggest-only by default.
- **No unprotected protected columns** — byte-identity is asserted physically
  after every run.

See `docs/semantic-models.md` for install, air-gapped setup, calibration, and
troubleshooting.
